In [1]:
clean_up = True # if True, remove all gams related files from working folder before starting
%run stdPackages.ipynb

The file _gams_py_gdb0.gdx is still active and was not deleted.
The file _gams_py_gdb1.gdx is still active and was not deleted.


*Load specific modules used here:*

In [2]:
os.chdir(d['py'])
import mWaste
from gmsPython import nestingTree
from valueShares import InitTree, nestedShares

# Waste management module

Load data:

In [3]:
t0 = 2019
v = 'vGRS'
db = GpyDB(os.path.join(d['data'], f'{v}{t0}_db'), name = f'{v}{t0}_db', ws = d['work'])
ws = db.ws

For variables that are defined over $t$, but where we do not yet have an initial value for all $t$, extrapolate from data:

*Note: This forces extrapolation of all variables defined over $t$ - if it is important that some variables are not extrapolated, they should be removed from this statement.*

In [4]:
[symbol.__setattr__('vals', extrapolateUpper(symbol.vals, db('tE')[0])) for symbol in [db[k] for k in db.varDom('t')['t']]];

Subset data to only include production sectors, but keep a full copy:

In [5]:
db0 = db.copy()
sIdx = db('sWaste')
AggDB.subsetDB(db, sIdx)

## 1. Nesting structure

*Temporarily add a branch to the output structure: This is to make sure that the ZW node is designated an intermediate input.*

In [6]:
outTree = db('nestWasteOut').union(pd.MultiIndex.from_arrays([db('sWaste'), ['DELBRANCH'], ['ZW']], names = ['s','n','nn']))
nestInp = nestingTree.Tree('Inp', tree = db('nestWasteInp').to_list(), f = 'CES_scaled')
nestOut = nestingTree.Tree('Out', tree = outTree.to_list(), f= 'CET_scaled')
nestAgg = nestingTree.AggTree(name = 'W', trees = {t.name: t for t in [nestInp, nestOut]}, ws = ws)
nestAgg(namespace = {str(n)+'_input': n for n in db('n').union(db('nm_D'))})

*Add static user cost as initial value for prices on durables:*

In [7]:
db.aom(db('pD_dur'), name = 'pD', priority = 'first')

For the waste management firm, the production structure of the intermediate good ```KELM``` is straightforward. However, when this is split into the generic service ```Waste``` and ```ZW``` (treatment aggregate), we deviate from the standard production module methods. We now do the following:
* Remove ```DELBRANCH``` from nesting structure: This is only included to make sure that ```ZW``` is designated an "intermediate" input and assigned to the correct subsets/mappings.
* We set the price of ```ZW = 1``` and compute quantity from the waste treatment production function.
* We know the quantity supplied of the residual service ```Waste```. Given that the production structure is CRS, we then back out an initial guess of a cost-index of this ```Waste``` output.
* Given these inputs, we can use "standard" methods to get share parameters, prices, and quantities for the nesting structure that solves the model in the baseline. 
* Remove ```ZW``` from the set of "intermediate goods": This subset is used to guide what prices/quantities are endogenous/exogenous in what solution states. We need new definitions for this specific group. Furthermore, remove ```ZW``` from the set of knots that require a price index equation (outputs, knots):

*Remove DELBRANCH from nesting trees:*

In [8]:
delIdx = pd.Index(['DELBRANCH'], name = 'n')
[nestAgg.db.__setitem__(k, adj.rc_pd(nestAgg.db(k), ('not', delIdx))) for k in nestAgg.db.varDom('n', types = ['set','subset','map'])['n']]; 
[nestOut.db[k].__setattr__('vals', adj.rc_pd(nestOut.db[k], ('not', delIdx))) for k in nestOut.db if 'n' in nestOut.db[k].domains];
delIdx = pd.Index(['DELBRANCH'], name = 'nn')
[nestAgg.db.__setitem__(k, adj.rc_pd(nestAgg.db(k), ('not', delIdx))) for k in nestAgg.db.varDom('n', types = ['set','subset','map'])['nn']];
[nestOut.db[k].__setattr__('vals', adj.rc_pd(nestOut.db[k], ('not', delIdx))) for k in nestOut.db if 'nn' in nestOut.db[k].domains];

*Get initial values*

In [9]:
v = InitTree(nestAgg, ws = ws)

*Price target of 1 in $ZW$ nest*

In [10]:
zwIdx = adj.rc_pd(nestAgg.get('int'), db('n_ZW'))
pDTemp = pd.Series(1, index = zwIdx)

*Production of $ZW$ from waste treatment production function:*

In [11]:
qZW = (db('WTD_W') * (db('WTD_gd') * db('WTD_d') + db('WTD_ge') * db('WTD_e') + db('WTD_gr') * db('WTD_r')*(1-db('WTD_dmin')))).xs(t0).sum()
qDTemp = pd.Series(qZW, index = zwIdx)

*Production of residual service, back out balancing price index:*

In [12]:
qS, qD, pD = v.defaultQS(db), v.defaultQD(db), v.defaultPD(db)
pS = ((qD*pD).groupby('s').sum() - qZW) / qS

*Get initial values of $\mu$-parameters and intermediate goods:*

In [13]:
qD = qDTemp.combine_first(qD)
pD = pDTemp.combine_first(pD)
v.db['sigma'] = adj.rc_pd(db('sigma'), v.db('s'))
v.db['eta'] = adj.rc_pd(db('eta'), v.db('s'))
vals = v(dbIO = db, qD = qD, pD = pD, qS = qS, pS = pS, balancePS = False)

*Get normalization parameters as well:*

In [14]:
qNorms = v.get_qNorm(vals)

*Replace definition of outputs, remove ```ZW``` from a few sets:*

In [15]:
nestAgg.db[nestAgg.n('output')] = adj.rc_pd(db('qS'), db('sWaste')).xs(t0).index.remove_unused_levels()
[nestAgg.db[nestAgg.n(k)].__setattr__('vals', adj.rc_pd(nestAgg.get(k), ('not', db('n_ZW')))) for k in ('int','knout')];
[nestAgg.db[nestAgg.n(k, local = 'Out')].__setattr__('vals', adj.rc_pd(nestAgg.get(k,local='Out'), ('not', db('n_ZW')))) for k in ['knot']];

*Add to main database:*

In [16]:
[db.aom(stdSort(adjMultiIndex.bc(vals[k], db('txE'))), name = k, priority = 'second') for k in ('pD','qD','pS','qS')];
[db.aom(gpy(stdSort(adjMultiIndex.bc(qNorms[k], db('txE'))), type = 'par', name = k)) for k in ('qDnorm','qSnorm')];
db.aom(vals['mu'], name = 'mu', priority = 'second')

## 2. Init and solve

Initialize model instance:

In [17]:
[nestAgg.db.__setitem__(k, db(k)) for k in ('dWTn','dWTy','dWTyF','n_ZW')];
m = mWaste.DynamicNCES_emission(nestAgg, exoP = db('d_pEqui'), exoQS = db('d_qSEqui'), partial = True, initFromGms = 'initFirmValueBlock')

*Test: Use subsetted database. Add other symbols specifically where it's needed for the waste treatment module.*

In [18]:
[m.db.aom(db0(k), name = k, priority = 'second') for k in ('dWS','qWS','uWS_D')];
m.initStuff(db = db)
AggDB.readSets(m.db)
m.db.mergeInternal()

Solve calibration:

In [19]:
solveDB = m.solve(state='C')

Add solution to main remove local init methods:

In [20]:
[m.db.__setitem__(k, solveDB[k]) for k in m.db.getTypes(['var']) if k in solveDB.symbols]; # use solution database
m.db.mergeInternal()
m.initFromGms = None # no longer use gams code to initialize variables

Test solve baseline version as well:

In [21]:
solB = m.solve(state = 'B')